In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置接口 (在此处修改所需运行的阶段)
# ==========================================
TARGET_STAGE = 'P1'  # 可选参数: 'P1', 'P2', 'P3', 'P4', 'P5'
INPUT_DIR = "./engineered_features"
N_TRIALS = 15        # 贝叶斯优化迭代次数
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def optimize_params(X, y, groups, n_trials=15):
    """
    内部优化器：目标函数已修改为最小化 MAPE
    """
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 150, step=50),
            'max_depth': trial.suggest_int('max_depth', 10, 25),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 8),
            'max_features': trial.suggest_float('max_features', 0.4, 0.8),
            'random_state': 42,
            'n_jobs': -1
        }
        rf = RandomForestRegressor(**params)
        logo = LeaveOneGroupOut()
        
        mape_list = []
        for train_idx, val_idx in logo.split(X, y, groups=groups):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            rf.fit(X_tr, y_tr)
            preds = rf.predict(X_val)
            
            # 计算当前交叉验证折的 MAPE
            fold_mape = np.mean(np.abs((y_val - preds) / (y_val + 1e-8))) * 100
            mape_list.append(fold_mape)
            
        # 返回平均 MAPE 供 Optuna 最小化
        return np.mean(mape_list)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

def perform_nested_loyo_cv(df, feature_cols, target_col='yield', year_col='Year'):
    years = sorted(df[year_col].unique())
    all_y_true, all_y_pred = [], []
    stage_results = []
    
    for test_year in years:
        train_df = df[df[year_col] != test_year]
        test_df = df[df[year_col] == test_year]
        
        X_train, y_train = train_df[feature_cols], train_df[target_col]
        groups_train = train_df[year_col]
        X_test, y_test = test_df[feature_cols], test_df[target_col]
        
        best_params = optimize_params(X_train, y_train, groups_train, n_trials=N_TRIALS)
        
        rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_test)
        
        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        
        m = calculate_metrics(y_test.values, y_pred)
        m['Test_Year'] = str(test_year)
        m['Best_Params'] = str(best_params)
        stage_results.append(m)
        
        print(f"    -> [Year {test_year}] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
        
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall'
    global_metrics['Best_Params'] = 'N/A'
    stage_results.append(global_metrics)
    
    return stage_results

def run_single_stage_training(input_dir, target_stage):
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        print("未找到 CSV 文件，请检查路径。")
        return
        
    df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    
    static_features = ['Elevation', 'Slope', 'Aspect', 'Sand', 'Clay', 'BD', 'pH', 'SOC']
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    all_columns = df.columns.tolist()
    
    periods = ['P1', 'P2', 'P3', 'P4', 'P5']
    if target_stage not in periods:
        print(f"无效的阶段目标: {target_stage}。必须为 P1 到 P5 之一。")
        return
        
    stage_idx = periods.index(target_stage)
    active_periods = periods[:stage_idx+1]
    
    dynamic_features = [col for col in all_columns if any(p in col for p in active_periods) and col not in meta_cols and col not in static_features]
    current_features = static_features + dynamic_features
    
    print(f"\n>>> 独立执行阶段: {target_stage} (包含特征期: {' + '.join(active_periods)})")
    print(f"    特征总数: {len(current_features)}")
    print(f"    优化目标: 最小化 MAPE")
    
    all_results = []
    stage_metrics_list = perform_nested_loyo_cv(df, current_features)
    
    for metric_dict in stage_metrics_list:
        metric_dict['Stage'] = target_stage
        all_results.append(metric_dict)
        if metric_dict['Test_Year'] == 'Overall':
            m = metric_dict
            print(f"\n    [{target_stage} 总体汇总] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}\n")

    cols = ['Stage', 'Test_Year', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    results_df = pd.DataFrame(all_results)[cols]
    
    # 动态命名输出文件，防止覆盖
    output_filename = f"Nested_Result_Stage_{target_stage}.csv"
    results_df.to_csv(output_filename, index=False)
    print(f"评估完成。当前阶段 ({target_stage}) 指标记录在 {output_filename} 中。")

if __name__ == "__main__":
    run_single_stage_training(INPUT_DIR, TARGET_STAGE)

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置接口 (在此处修改所需运行的阶段)
# ==========================================
TARGET_STAGE = 'P2'  # 可选参数: 'P1', 'P2', 'P3', 'P4', 'P5'
INPUT_DIR = "./engineered_features"
N_TRIALS = 15        # 贝叶斯优化迭代次数
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def optimize_params(X, y, groups, n_trials=15):
    """
    内部优化器：目标函数已修改为最小化 MAPE
    """
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 150, step=50),
            'max_depth': trial.suggest_int('max_depth', 10, 25),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 8),
            'max_features': trial.suggest_float('max_features', 0.4, 0.8),
            'random_state': 42,
            'n_jobs': -1
        }
        rf = RandomForestRegressor(**params)
        logo = LeaveOneGroupOut()
        
        mape_list = []
        for train_idx, val_idx in logo.split(X, y, groups=groups):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            rf.fit(X_tr, y_tr)
            preds = rf.predict(X_val)
            
            # 计算当前交叉验证折的 MAPE
            fold_mape = np.mean(np.abs((y_val - preds) / (y_val + 1e-8))) * 100
            mape_list.append(fold_mape)
            
        # 返回平均 MAPE 供 Optuna 最小化
        return np.mean(mape_list)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

def perform_nested_loyo_cv(df, feature_cols, target_col='yield', year_col='Year'):
    years = sorted(df[year_col].unique())
    all_y_true, all_y_pred = [], []
    stage_results = []
    
    for test_year in years:
        train_df = df[df[year_col] != test_year]
        test_df = df[df[year_col] == test_year]
        
        X_train, y_train = train_df[feature_cols], train_df[target_col]
        groups_train = train_df[year_col]
        X_test, y_test = test_df[feature_cols], test_df[target_col]
        
        best_params = optimize_params(X_train, y_train, groups_train, n_trials=N_TRIALS)
        
        rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_test)
        
        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        
        m = calculate_metrics(y_test.values, y_pred)
        m['Test_Year'] = str(test_year)
        m['Best_Params'] = str(best_params)
        stage_results.append(m)
        
        print(f"    -> [Year {test_year}] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
        
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall'
    global_metrics['Best_Params'] = 'N/A'
    stage_results.append(global_metrics)
    
    return stage_results

def run_single_stage_training(input_dir, target_stage):
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        print("未找到 CSV 文件，请检查路径。")
        return
        
    df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    
    static_features = ['Elevation', 'Slope', 'Aspect', 'Sand', 'Clay', 'BD', 'pH', 'SOC']
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    all_columns = df.columns.tolist()
    
    periods = ['P1', 'P2', 'P3', 'P4', 'P5']
    if target_stage not in periods:
        print(f"无效的阶段目标: {target_stage}。必须为 P1 到 P5 之一。")
        return
        
    stage_idx = periods.index(target_stage)
    active_periods = periods[:stage_idx+1]
    
    dynamic_features = [col for col in all_columns if any(p in col for p in active_periods) and col not in meta_cols and col not in static_features]
    current_features = static_features + dynamic_features
    
    print(f"\n>>> 独立执行阶段: {target_stage} (包含特征期: {' + '.join(active_periods)})")
    print(f"    特征总数: {len(current_features)}")
    print(f"    优化目标: 最小化 MAPE")
    
    all_results = []
    stage_metrics_list = perform_nested_loyo_cv(df, current_features)
    
    for metric_dict in stage_metrics_list:
        metric_dict['Stage'] = target_stage
        all_results.append(metric_dict)
        if metric_dict['Test_Year'] == 'Overall':
            m = metric_dict
            print(f"\n    [{target_stage} 总体汇总] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}\n")

    cols = ['Stage', 'Test_Year', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    results_df = pd.DataFrame(all_results)[cols]
    
    # 动态命名输出文件，防止覆盖
    output_filename = f"Nested_Result_Stage_{target_stage}.csv"
    results_df.to_csv(output_filename, index=False)
    print(f"评估完成。当前阶段 ({target_stage}) 指标记录在 {output_filename} 中。")

if __name__ == "__main__":
    run_single_stage_training(INPUT_DIR, TARGET_STAGE)

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置接口 (在此处修改所需运行的阶段)
# ==========================================
TARGET_STAGE = 'P3'  # 可选参数: 'P1', 'P2', 'P3', 'P4', 'P5'
INPUT_DIR = "./engineered_features"
N_TRIALS = 15        # 贝叶斯优化迭代次数
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def optimize_params(X, y, groups, n_trials=15):
    """
    内部优化器：目标函数已修改为最小化 MAPE
    """
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 150, step=50),
            'max_depth': trial.suggest_int('max_depth', 10, 25),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 8),
            'max_features': trial.suggest_float('max_features', 0.4, 0.8),
            'random_state': 42,
            'n_jobs': -1
        }
        rf = RandomForestRegressor(**params)
        logo = LeaveOneGroupOut()
        
        mape_list = []
        for train_idx, val_idx in logo.split(X, y, groups=groups):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            rf.fit(X_tr, y_tr)
            preds = rf.predict(X_val)
            
            # 计算当前交叉验证折的 MAPE
            fold_mape = np.mean(np.abs((y_val - preds) / (y_val + 1e-8))) * 100
            mape_list.append(fold_mape)
            
        # 返回平均 MAPE 供 Optuna 最小化
        return np.mean(mape_list)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

def perform_nested_loyo_cv(df, feature_cols, target_col='yield', year_col='Year'):
    years = sorted(df[year_col].unique())
    all_y_true, all_y_pred = [], []
    stage_results = []
    
    for test_year in years:
        train_df = df[df[year_col] != test_year]
        test_df = df[df[year_col] == test_year]
        
        X_train, y_train = train_df[feature_cols], train_df[target_col]
        groups_train = train_df[year_col]
        X_test, y_test = test_df[feature_cols], test_df[target_col]
        
        best_params = optimize_params(X_train, y_train, groups_train, n_trials=N_TRIALS)
        
        rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_test)
        
        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        
        m = calculate_metrics(y_test.values, y_pred)
        m['Test_Year'] = str(test_year)
        m['Best_Params'] = str(best_params)
        stage_results.append(m)
        
        print(f"    -> [Year {test_year}] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
        
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall'
    global_metrics['Best_Params'] = 'N/A'
    stage_results.append(global_metrics)
    
    return stage_results

def run_single_stage_training(input_dir, target_stage):
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        print("未找到 CSV 文件，请检查路径。")
        return
        
    df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    
    static_features = ['Elevation', 'Slope', 'Aspect', 'Sand', 'Clay', 'BD', 'pH', 'SOC']
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    all_columns = df.columns.tolist()
    
    periods = ['P1', 'P2', 'P3', 'P4', 'P5']
    if target_stage not in periods:
        print(f"无效的阶段目标: {target_stage}。必须为 P1 到 P5 之一。")
        return
        
    stage_idx = periods.index(target_stage)
    active_periods = periods[:stage_idx+1]
    
    dynamic_features = [col for col in all_columns if any(p in col for p in active_periods) and col not in meta_cols and col not in static_features]
    current_features = static_features + dynamic_features
    
    print(f"\n>>> 独立执行阶段: {target_stage} (包含特征期: {' + '.join(active_periods)})")
    print(f"    特征总数: {len(current_features)}")
    print(f"    优化目标: 最小化 MAPE")
    
    all_results = []
    stage_metrics_list = perform_nested_loyo_cv(df, current_features)
    
    for metric_dict in stage_metrics_list:
        metric_dict['Stage'] = target_stage
        all_results.append(metric_dict)
        if metric_dict['Test_Year'] == 'Overall':
            m = metric_dict
            print(f"\n    [{target_stage} 总体汇总] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}\n")

    cols = ['Stage', 'Test_Year', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    results_df = pd.DataFrame(all_results)[cols]
    
    # 动态命名输出文件，防止覆盖
    output_filename = f"Nested_Result_Stage_{target_stage}.csv"
    results_df.to_csv(output_filename, index=False)
    print(f"评估完成。当前阶段 ({target_stage}) 指标记录在 {output_filename} 中。")

if __name__ == "__main__":
    run_single_stage_training(INPUT_DIR, TARGET_STAGE)

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置接口 (在此处修改所需运行的阶段)
# ==========================================
TARGET_STAGE = 'P4'  # 可选参数: 'P1', 'P2', 'P3', 'P4', 'P5'
INPUT_DIR = "./engineered_features"
N_TRIALS = 15        # 贝叶斯优化迭代次数
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def optimize_params(X, y, groups, n_trials=15):
    """
    内部优化器：目标函数已修改为最小化 MAPE
    """
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 150, step=50),
            'max_depth': trial.suggest_int('max_depth', 10, 25),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 8),
            'max_features': trial.suggest_float('max_features', 0.4, 0.8),
            'random_state': 42,
            'n_jobs': -1
        }
        rf = RandomForestRegressor(**params)
        logo = LeaveOneGroupOut()
        
        mape_list = []
        for train_idx, val_idx in logo.split(X, y, groups=groups):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            rf.fit(X_tr, y_tr)
            preds = rf.predict(X_val)
            
            # 计算当前交叉验证折的 MAPE
            fold_mape = np.mean(np.abs((y_val - preds) / (y_val + 1e-8))) * 100
            mape_list.append(fold_mape)
            
        # 返回平均 MAPE 供 Optuna 最小化
        return np.mean(mape_list)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

def perform_nested_loyo_cv(df, feature_cols, target_col='yield', year_col='Year'):
    years = sorted(df[year_col].unique())
    all_y_true, all_y_pred = [], []
    stage_results = []
    
    for test_year in years:
        train_df = df[df[year_col] != test_year]
        test_df = df[df[year_col] == test_year]
        
        X_train, y_train = train_df[feature_cols], train_df[target_col]
        groups_train = train_df[year_col]
        X_test, y_test = test_df[feature_cols], test_df[target_col]
        
        best_params = optimize_params(X_train, y_train, groups_train, n_trials=N_TRIALS)
        
        rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_test)
        
        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        
        m = calculate_metrics(y_test.values, y_pred)
        m['Test_Year'] = str(test_year)
        m['Best_Params'] = str(best_params)
        stage_results.append(m)
        
        print(f"    -> [Year {test_year}] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
        
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall'
    global_metrics['Best_Params'] = 'N/A'
    stage_results.append(global_metrics)
    
    return stage_results

def run_single_stage_training(input_dir, target_stage):
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        print("未找到 CSV 文件，请检查路径。")
        return
        
    df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    
    static_features = ['Elevation', 'Slope', 'Aspect', 'Sand', 'Clay', 'BD', 'pH', 'SOC']
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    all_columns = df.columns.tolist()
    
    periods = ['P1', 'P2', 'P3', 'P4', 'P5']
    if target_stage not in periods:
        print(f"无效的阶段目标: {target_stage}。必须为 P1 到 P5 之一。")
        return
        
    stage_idx = periods.index(target_stage)
    active_periods = periods[:stage_idx+1]
    
    dynamic_features = [col for col in all_columns if any(p in col for p in active_periods) and col not in meta_cols and col not in static_features]
    current_features = static_features + dynamic_features
    
    print(f"\n>>> 独立执行阶段: {target_stage} (包含特征期: {' + '.join(active_periods)})")
    print(f"    特征总数: {len(current_features)}")
    print(f"    优化目标: 最小化 MAPE")
    
    all_results = []
    stage_metrics_list = perform_nested_loyo_cv(df, current_features)
    
    for metric_dict in stage_metrics_list:
        metric_dict['Stage'] = target_stage
        all_results.append(metric_dict)
        if metric_dict['Test_Year'] == 'Overall':
            m = metric_dict
            print(f"\n    [{target_stage} 总体汇总] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}\n")

    cols = ['Stage', 'Test_Year', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    results_df = pd.DataFrame(all_results)[cols]
    
    # 动态命名输出文件，防止覆盖
    output_filename = f"Nested_Result_Stage_{target_stage}.csv"
    results_df.to_csv(output_filename, index=False)
    print(f"评估完成。当前阶段 ({target_stage}) 指标记录在 {output_filename} 中。")

if __name__ == "__main__":
    run_single_stage_training(INPUT_DIR, TARGET_STAGE)

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置接口 (在此处修改所需运行的阶段)
# ==========================================
TARGET_STAGE = 'P5'  # 可选参数: 'P1', 'P2', 'P3', 'P4', 'P5'
INPUT_DIR = "./engineered_features"
N_TRIALS = 15        # 贝叶斯优化迭代次数
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def optimize_params(X, y, groups, n_trials=15):
    """
    内部优化器：目标函数已修改为最小化 MAPE
    """
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 150, step=50),
            'max_depth': trial.suggest_int('max_depth', 10, 25),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 8),
            'max_features': trial.suggest_float('max_features', 0.4, 0.8),
            'random_state': 42,
            'n_jobs': -1
        }
        rf = RandomForestRegressor(**params)
        logo = LeaveOneGroupOut()
        
        mape_list = []
        for train_idx, val_idx in logo.split(X, y, groups=groups):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            rf.fit(X_tr, y_tr)
            preds = rf.predict(X_val)
            
            # 计算当前交叉验证折的 MAPE
            fold_mape = np.mean(np.abs((y_val - preds) / (y_val + 1e-8))) * 100
            mape_list.append(fold_mape)
            
        # 返回平均 MAPE 供 Optuna 最小化
        return np.mean(mape_list)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

def perform_nested_loyo_cv(df, feature_cols, target_col='yield', year_col='Year'):
    years = sorted(df[year_col].unique())
    all_y_true, all_y_pred = [], []
    stage_results = []
    
    for test_year in years:
        train_df = df[df[year_col] != test_year]
        test_df = df[df[year_col] == test_year]
        
        X_train, y_train = train_df[feature_cols], train_df[target_col]
        groups_train = train_df[year_col]
        X_test, y_test = test_df[feature_cols], test_df[target_col]
        
        best_params = optimize_params(X_train, y_train, groups_train, n_trials=N_TRIALS)
        
        rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_test)
        
        all_y_true.extend(y_test.values)
        all_y_pred.extend(y_pred)
        
        m = calculate_metrics(y_test.values, y_pred)
        m['Test_Year'] = str(test_year)
        m['Best_Params'] = str(best_params)
        stage_results.append(m)
        
        print(f"    -> [Year {test_year}] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
        
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall'
    global_metrics['Best_Params'] = 'N/A'
    stage_results.append(global_metrics)
    
    return stage_results

def run_single_stage_training(input_dir, target_stage):
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        print("未找到 CSV 文件，请检查路径。")
        return
        
    df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    
    static_features = ['Elevation', 'Slope', 'Aspect', 'Sand', 'Clay', 'BD', 'pH', 'SOC']
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    all_columns = df.columns.tolist()
    
    periods = ['P1', 'P2', 'P3', 'P4', 'P5']
    if target_stage not in periods:
        print(f"无效的阶段目标: {target_stage}。必须为 P1 到 P5 之一。")
        return
        
    stage_idx = periods.index(target_stage)
    active_periods = periods[:stage_idx+1]
    
    dynamic_features = [col for col in all_columns if any(p in col for p in active_periods) and col not in meta_cols and col not in static_features]
    current_features = static_features + dynamic_features
    
    print(f"\n>>> 独立执行阶段: {target_stage} (包含特征期: {' + '.join(active_periods)})")
    print(f"    特征总数: {len(current_features)}")
    print(f"    优化目标: 最小化 MAPE")
    
    all_results = []
    stage_metrics_list = perform_nested_loyo_cv(df, current_features)
    
    for metric_dict in stage_metrics_list:
        metric_dict['Stage'] = target_stage
        all_results.append(metric_dict)
        if metric_dict['Test_Year'] == 'Overall':
            m = metric_dict
            print(f"\n    [{target_stage} 总体汇总] R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}\n")

    cols = ['Stage', 'Test_Year', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    results_df = pd.DataFrame(all_results)[cols]
    
    # 动态命名输出文件，防止覆盖
    output_filename = f"Nested_Result_Stage_{target_stage}.csv"
    results_df.to_csv(output_filename, index=False)
    print(f"评估完成。当前阶段 ({target_stage}) 指标记录在 {output_filename} 中。")

if __name__ == "__main__":
    run_single_stage_training(INPUT_DIR, TARGET_STAGE)